# 03 — CNN on Mel-Spectrograms

We treat each audio clip as a 2-D image: the mel-spectrogram (128 mel bands × T time frames).
A 4-block CNN learns to recognise emotion from these spectral patterns.

In [ ]:
import sys; sys.path.insert(0, '..')

import os, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from addict import Dict

from data_classes.ravdess_dataset import RAVDESSDataset
from model_classes.cnn_model import CNNEmotionClassifier
from utils import set_seed, compute_metrics, plot_confusion_matrix, plot_training_curves, print_report

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

with open('../config/default.yaml') as f:
    cfg = Dict(yaml.safe_load(f))

set_seed(cfg.training.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Dataset & DataLoaders

In [ ]:
test_actors      = list(cfg.data.test_actors)
train_val_actors = [a for a in range(1, 25) if a not in test_actors]

ds_args = dict(
    mode='melspec',
    sample_rate=cfg.data.sample_rate, duration=cfg.data.duration,
    n_mfcc=cfg.data.n_mfcc, n_mels=cfg.data.n_mels,
    n_fft=cfg.data.n_fft, hop_length=cfg.data.hop_length,
)

full_ds = RAVDESSDataset(cfg.data.data_dir, actor_ids=train_val_actors, **ds_args)
test_ds = RAVDESSDataset(cfg.data.data_dir, actor_ids=test_actors,      **ds_args)

val_n, train_n = int(0.15*len(full_ds)), len(full_ds) - int(0.15*len(full_ds))
train_ds, val_ds = random_split(full_ds, [train_n, val_n],
                                 generator=torch.Generator().manual_seed(cfg.training.seed))

# Show a spectrogram sample
x, label = train_ds[0]
print(f'Spectrogram shape: {x.shape}   Label: {label}')
plt.figure(figsize=(8, 3))
plt.imshow(x[0].numpy(), aspect='auto', origin='lower', cmap='magma')
plt.colorbar(label='Normalised power')
plt.title('Example Mel-Spectrogram')
plt.xlabel('Time frame'); plt.ylabel('Mel band')
plt.tight_layout(); plt.show()

## 2. Model Architecture

In [ ]:
model = CNNEmotionClassifier(n_classes=cfg.data.n_classes,
                              n_mels=cfg.data.n_mels,
                              dropout=cfg.model.cnn.dropout).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal parameters: {total_params:,}')

## 3. Training

In [ ]:
nw = 0  # Windows: 0 workers
train_loader = DataLoader(train_ds, cfg.training.batch_size, shuffle=True,  num_workers=nw)
val_loader   = DataLoader(val_ds,   cfg.training.batch_size, shuffle=False, num_workers=nw)
test_loader  = DataLoader(test_ds,  cfg.training.batch_size, shuffle=False, num_workers=nw)

weights   = full_ds.class_weights().to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.training.learning_rate,
                              weight_decay=cfg.training.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=6, factor=0.5)

os.makedirs('../saved_models', exist_ok=True)
os.makedirs('../results', exist_ok=True)

best_val_acc = 0
patience_cnt = 0
tr_losses, vl_losses, tr_accs, vl_accs = [], [], [], []

for epoch in range(1, cfg.training.epochs + 1):
    # Train
    model.train()
    tl = tc = tt = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        tl += loss.item()*len(y); tc += (out.argmax(1)==y).sum().item(); tt += len(y)
    tr_losses.append(tl/tt); tr_accs.append(tc/tt)

    # Val
    model.eval()
    vl = vc = vt = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            vl += loss.item()*len(y); vc += (out.argmax(1)==y).sum().item(); vt += len(y)
    vl_losses.append(vl/vt); vl_accs.append(vc/vt)
    scheduler.step(vl/vt)

    flag = ''
    if vl_accs[-1] > best_val_acc:
        best_val_acc = vl_accs[-1]
        patience_cnt = 0
        torch.save(model.state_dict(), '../saved_models/best_cnn.pth')
        flag = ' ✓'
    else:
        patience_cnt += 1

    if epoch % 5 == 0 or flag:
        print(f'Epoch {epoch:03d}  train_loss={tr_losses[-1]:.4f} train_acc={tr_accs[-1]:.4f}  '
              f'val_loss={vl_losses[-1]:.4f} val_acc={vl_accs[-1]:.4f}{flag}')

    if patience_cnt >= cfg.training.early_stopping_patience:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest val accuracy: {best_val_acc:.4f}')

## 4. Training Curves

In [ ]:
fig = plot_training_curves(tr_losses, vl_losses, tr_accs, vl_accs,
                            save_path='../results/curves_cnn.png')
plt.show()

## 5. Test Evaluation

In [ ]:
model.load_state_dict(torch.load('../saved_models/best_cnn.pth', map_location=device))
model.eval()
preds, labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(y.numpy())

preds, labels = np.array(preds), np.array(labels)
metrics = compute_metrics(labels, preds)
print(f'Accuracy  : {metrics["accuracy"]:.4f}')
print(f'F1 macro  : {metrics["f1_macro"]:.4f}')
print(f'F1 weighted: {metrics["f1_weighted"]:.4f}')
print_report(labels, preds)

json.dump(metrics, open('../results/metrics_cnn.json','w'), indent=2)

## 6. Confusion Matrix

In [ ]:
fig = plot_confusion_matrix(labels, preds, title='CNN — Confusion Matrix',
                             save_path='../results/cm_cnn.png')
plt.show()